# Part 1. Source Integration and Internal Schema

## Part Scope

This layer converts SEC source records into a stable internal annual schema for downstream analysis.

The main output is not a raw filing response.  
The output is a year-specific snapshot model that preserves both analytical values and field-level source traceability.

That internal structure becomes the base unit for later metric computation, interpretation, storage, and delivery.

## Source Retrieval Boundary

Source retrieval begins with SEC company data.

A user-provided ticker is normalized and resolved into a CIK identifier.  
The CIK is then used to access company-level filing facts from the SEC companyfacts endpoint, which defines the raw source boundary for this workflow.

This separation keeps source access logic distinct from the transformation logic used to build internal records.

In [ ]:
SEC_HEADERS = {
    "User-Agent": "name your_email@example.com",
    "Accept-Encoding": "gzip, deflate",
}

TICKER_CIK_URL = "https://www.sec.gov/files/company_tickers.json"
COMPANYFACTS_URL = "https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json"


def _normalize_ticker(ticker: str) -> str:
    return ticker.strip().upper()


def resolve_cik_from_ticker(ticker: str) -> Optional[str]:
    ticker = _normalize_ticker(ticker)
    data = _safe_get_json(TICKER_CIK_URL, headers=SEC_HEADERS)
    if not data:
        return None

    for _, row in data.items():
        if str(row.get("ticker", "")).upper() == ticker:
            cik_int = row.get("cik_str")
            if cik_int is None:
                return None
            return str(cik_int).zfill(10)

    return None

## Annual Record Construction

Raw SEC responses are not used as direct analytical inputs.

Instead, filing facts are filtered by fiscal year and reconstructed into annual records built around a fixed internal field set.  
The record includes income statement, cash flow, balance sheet, and per-share fields required for later comparison.

This reconstruction step turns heterogeneous filing structures into comparable year-level objects.

In [ ]:
def fetch_single_10k(ticker: str, year: int) -> Optional[FinancialSnapshot]:
    normalized_ticker = _normalize_ticker(ticker)
    cik = resolve_cik_from_ticker(normalized_ticker)
    if not cik:
        return None

    url = COMPANYFACTS_URL.format(cik=cik)
    data = _safe_get_json(url, headers=SEC_HEADERS)
    if not data:
        return None

    facts = data.get("facts", {}).get("us-gaap", {})

    revenue, revenue_meta = _extract_latest_value_for_year_with_source(
        facts,
        tags=["Revenues", "RevenueFromContractWithCustomerExcludingAssessedTax", "SalesRevenueNet"],
        fiscal_year=year,
    )
    net_income, net_income_meta = _extract_latest_value_for_year_with_source(
        facts,
        tags=["NetIncomeLoss"],
        fiscal_year=year,
    )
    operating_income, operating_income_meta = _extract_latest_value_for_year_with_source(
        facts,
        tags=["OperatingIncomeLoss"],
        fiscal_year=year,
    )
    operating_cash_flow, operating_cash_flow_meta = _extract_latest_value_for_year_with_source(
        facts,
        tags=["NetCashProvidedByUsedInOperatingActivities"],
        fiscal_year=year,
    )

## Field Mapping Strategy

Internal fields are populated through tag-group mapping rather than one-to-one dependence on a single SEC label.

For several concepts, multiple candidate tags are evaluated and the latest valid value for the target fiscal year is selected.  
This approach preserves analytical continuity even when equivalent concepts appear under different labels across filings.

The mapping layer is therefore designed around semantic consistency at the internal model level, not around rigid dependence on one raw field name.

In [ ]:
def _extract_latest_value_for_year_with_source(
    facts: dict,
    tags: List[str],
    fiscal_year: int,
    unit: str = "USD",
) -> Tuple[Optional[float], Dict[str, Any]]:
    for tag in tags:
        item = facts.get(tag, {})
        units = item.get("units", {})
        rows = units.get(unit, [])

        year_rows = [r for r in rows if r.get("fy") == fiscal_year]
        if not year_rows:
            continue

        year_rows = sorted(
            year_rows,
            key=lambda x: (x.get("filed", ""), x.get("end", "")),
            reverse=True,
        )

        for row in year_rows:
            val = row.get("val")
            if val is not None:
                return val, {
                    "tag": tag,
                    "unit": unit,
                    "fy": fiscal_year,
                    "filed": row.get("filed"),
                    "end": row.get("end"),
                    "frame": row.get("frame"),
                    "form": row.get("form"),
                    "source_type": "direct_tag_match",
                }

    return None, {
        "tag": None,
        "unit": unit,
        "fy": fiscal_year,
        "source_type": "missing",
    }

## Internal Snapshot Model

The core internal object in this layer is the annual snapshot model.

Each snapshot stores normalized financial fields together with retrieval metadata such as ticker, CIK, source URL, retrieval timestamp, and field-level source records.  
That design keeps analytical values and lineage information inside the same object.

Downstream layers therefore operate on a stable record structure without losing traceability back to the original source tags.

In [ ]:
class FinancialSnapshot:
    fiscal_year: int

    # Income statement
    revenue: Optional[float] = None
    net_income: Optional[float] = None
    operating_income: Optional[float] = None

    # Cash flow statement
    operating_cash_flow: Optional[float] = None
    capex: Optional[float] = None

    # Balance sheet
    cash_and_equivalents: Optional[float] = None
    total_debt: Optional[float] = None
    total_equity: Optional[float] = None
    total_assets: Optional[float] = None
    total_liabilities: Optional[float] = None
    current_assets: Optional[float] = None
    current_liabilities: Optional[float] = None

    # Per-share data
    shares_outstanding_basic: Optional[float] = None
    shares_outstanding_diluted: Optional[float] = None

    # Metadata for normalization / lineage / quality
    ticker: Optional[str] = None
    cik: Optional[str] = None
    source_url: Optional[str] = None
    retrieved_at: Optional[str] = None
    field_sources: Dict[str, Dict[str, Any]] = field(default_factory=dict)
    available_fields_count: int = 0
    missing_fields_count: int = 0
    minimal_validity: bool = False

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)

## Traceability and Coverage Control

Not every retrieved record is treated as analytically usable.

Each snapshot tracks field-level source metadata, available-field counts, missing-field counts, and a minimum validity condition before entering later stages.  
Records that fail the minimum condition are excluded from the normalized dataset.

This prevents structurally weak annual records from being treated as standard downstream inputs.

In [ ]:
snapshot = FinancialSnapshot(
    fiscal_year=year,
    revenue=revenue,
    net_income=net_income,
    operating_income=operating_income,
    operating_cash_flow=operating_cash_flow,
    capex=capex,
    cash_and_equivalents=cash_and_equivalents,
    total_debt=total_debt,
    total_equity=total_equity,
    total_assets=total_assets,
    total_liabilities=total_liabilities,
    current_assets=current_assets,
    current_liabilities=current_liabilities,
    shares_outstanding_basic=shares_basic,
    shares_outstanding_diluted=shares_diluted,
    ticker=normalized_ticker,
    cik=cik,
    source_url=url,
    retrieved_at=dt.datetime.utcnow().isoformat() + "Z",
    field_sources={
        "revenue": revenue_meta,
        "net_income": net_income_meta,
        "operating_income": operating_income_meta,
        "operating_cash_flow": operating_cash_flow_meta,
        "capex": capex_meta,
        "cash_and_equivalents": cash_meta,
        "total_debt": total_debt_meta,
        "total_equity": total_equity_meta,
        "total_assets": total_assets_meta,
        "total_liabilities": total_liabilities_meta,
        "current_assets": current_assets_meta,
        "current_liabilities": current_liabilities_meta,
        "shares_outstanding_basic": shares_basic_meta,
        "shares_outstanding_diluted": shares_diluted_meta,
    },
)

snapshot.available_fields_count = _count_available_fields(snapshot)
snapshot.missing_fields_count = _count_missing_fields(snapshot)
snapshot.minimal_validity = not (snapshot.revenue is None and snapshot.net_income is None)

if not snapshot.minimal_validity:
    return None

## Multi-Year Snapshot Collection

Once annual snapshots are defined, the workflow gathers multiple fiscal years and stores them as a chronologically ordered set.

This step transforms isolated source retrieval calls into a reusable longitudinal dataset.  
Later metric logic is therefore applied to normalized annual snapshots rather than to disconnected filing responses.

In [ ]:
def fetch_10k_snapshots(
    ticker: str,
    count: int = 5,
    current_year: Optional[int] = None,
) -> List[FinancialSnapshot]:
    if current_year is None:
        current_year = dt.datetime.now().year

    snapshots: List[FinancialSnapshot] = []

    for i in range(count):
        year = current_year - i
        snapshot = fetch_single_10k(ticker, year)
        if snapshot:
            snapshots.append(snapshot)

    return sorted(snapshots, key=lambda x: x.fiscal_year)

## Downstream Readiness

The output of this layer is a normalized annual dataset, not a raw API payload and not yet a derived KPI table.

Each record already contains the structure needed for metric generation, traceability checks, structured interpretation, and delivery-layer formatting.  
Source integration concerns are resolved here so that later layers can focus on transformation rather than reconstruction.

In [ ]:
def _snapshot_to_row(snapshot: Any, run_id: str, ticker: str, created_at: str) -> Dict[str, Any]:
    return {
        "run_id": run_id,
        "ticker": ticker,
        "created_at": created_at,
        "fiscal_year": getattr(snapshot, "fiscal_year", None),
        "revenue": getattr(snapshot, "revenue", None),
        "net_income": getattr(snapshot, "net_income", None),
        "operating_income": getattr(snapshot, "operating_income", None),
        "operating_cash_flow": getattr(snapshot, "operating_cash_flow", None),
        "capex": getattr(snapshot, "capex", None),
        "cash_and_equivalents": getattr(snapshot, "cash_and_equivalents", None),
        "total_debt": getattr(snapshot, "total_debt", None),
        "total_equity": getattr(snapshot, "total_equity", None),
        "source_url": getattr(snapshot, "source_url", None),
        "retrieved_at": getattr(snapshot, "retrieved_at", None),
        "cik": getattr(snapshot, "cik", None),
        "available_fields_count": getattr(snapshot, "available_fields_count", None),
        "missing_fields_count": getattr(snapshot, "missing_fields_count", None),
        "minimal_validity": int(bool(getattr(snapshot, "minimal_validity", False))),
        "field_sources_json": _safe_json_dumps(getattr(snapshot, "field_sources", {})),
    }

## Transition to the Next Layer

With the annual snapshot structure in place, the workflow moves from source normalization to analytical transformation.

Part 2 focuses on how normalized annual records are converted into comparable metrics for growth, profitability, cash conversion, and balance structure.